# Stage 3 Stratified Num Choices Task Training (A100)

Thin Colab orchestration notebook for training `c04_task_choice_stratified`, the f04 formulation with num-choices/task-stratified sampling.


## 1. Mount Google Drive

Mount Google Drive before running the repo bootstrap flow.


In [2]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


## 2. Configure Paths

Set the repo checkout path, Drive output directory, experiment ID, smoke size, and optional data override.


In [11]:
from pathlib import Path

REPO_URL = "https://github.com/Demetri65/dl-kaggle-competition-final.git"
REPO_REF = "main"
REPO_DIR = Path("/content/dl-kaggle-competition-final")
SOURCE_ENV = Path("/content/.env")
UPLOADER_KEY = "_env_uploader"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/p2p_runs/stage3_stratified_num_choices_task_a100"
DATA_DIR_OVERRIDE = ""
EXPERIMENT_ID = "c04_task_choice_stratified"
SMOKE_TRAIN_EXAMPLES = 128
SMOKE_VAL_EXAMPLES = 128
EVAL_BATCH_SIZE = 16
INFERENCE_COMPLETION_BATCH_SIZE = 16


## 3. Prepare The Repo

Upload `.env` if needed, sync the repo, and run `scripts/bootstrap_colab.sh`.


In [7]:
# -- 0. Colab setup ------------------------------------------------
import subprocess
import ipywidgets as widgets
from IPython.display import display


def get_uploaded_file(uploader):
    value = uploader.value

    if isinstance(value, dict):
        filename, uploaded_file = next(iter(value.items()))
        if isinstance(uploaded_file, dict):
            content = uploaded_file.get("content", uploaded_file.get("data"))
        else:
            content = uploaded_file
    else:
        uploaded_file = value[0]
        if isinstance(uploaded_file, dict):
            filename = uploaded_file["name"]
            content = uploaded_file["content"]
        else:
            filename = uploaded_file.name
            content = uploaded_file.content

    payload = content.tobytes() if hasattr(content, "tobytes") else bytes(content)
    return filename, payload


ready_to_bootstrap = SOURCE_ENV.exists()

if ready_to_bootstrap:
    print(f"Using existing {SOURCE_ENV}")
else:
    uploader = globals().get(UPLOADER_KEY)
    if uploader is None:
        uploader = widgets.FileUpload(accept=".env", multiple=False, description="Upload .env")
        globals()[UPLOADER_KEY] = uploader

    if not uploader.value:
        display(uploader)
        print("Select your local .env file in the upload widget above, then rerun this cell.")
    else:
        filename, payload = get_uploaded_file(uploader)
        SOURCE_ENV.write_bytes(payload)
        SOURCE_ENV.chmod(0o600)
        print(f"Saved {filename} to {SOURCE_ENV}")
        uploader.close()
        globals().pop(UPLOADER_KEY, None)
        ready_to_bootstrap = True

if ready_to_bootstrap:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"], check=True)

    print(f"Repo synced to latest origin/{REPO_REF} at {REPO_DIR}")
    result = subprocess.run(
        ["bash", "scripts/bootstrap_colab.sh"],
        cwd=REPO_DIR,
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout, end="")
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr, end="")
        raise RuntimeError(f"scripts/bootstrap_colab.sh failed with exit code {result.returncode}")


Saved .env to /content/.env
Repo synced to latest origin/main at /content/dl-kaggle-competition-final
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 55.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 29.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 212.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 168.5 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
 

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4. Helpers

Helpers route every run through the repo scripts and print the resolved config for verification.


In [9]:
import os
import shlex
import subprocess
from pathlib import Path

import yaml

REPO_ROOT = REPO_DIR.resolve()
Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT)

COMMON_OVERRIDES = []
if DATA_DIR_OVERRIDE:
    COMMON_OVERRIDES.append(f"data.data_dir={DATA_DIR_OVERRIDE}")


def with_common_overrides(overrides):
    return [*COMMON_OVERRIDES, *overrides]


def extend_with_overrides(args, overrides):
    for override in overrides:
        args.extend(["--set", override])


def run_repo_command(args):
    command = ["python3", *args]
    print("$", " ".join(shlex.quote(part) for part in command))
    result = subprocess.run(command, cwd=REPO_ROOT, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result


def find_resolved_config(run_root: Path, experiment_id: str) -> Path:
    candidates = sorted(run_root.glob(f"{experiment_id}_seed*/resolved_config.yaml"))
    if not candidates:
        raise FileNotFoundError(f"No resolved_config.yaml found under {run_root}")
    if len(candidates) > 1:
        print(f"Multiple resolved configs found; using {candidates[-1]}")
    return candidates[-1]


def verify_resolved_config(run_root: Path):
    path = find_resolved_config(run_root, EXPERIMENT_ID)
    config = yaml.safe_load(path.read_text())
    fields = [name for name, enabled in config["fields"].items() if enabled]
    print(f"Resolved config: {path}")
    print(f"experiment_id: {config['experiment_id']}")
    print(f"parent_experiment_id: {config.get('parent_experiment_id')}")
    print(f"formulation.mode: {config['formulation']['mode']}")
    print(f"prompting.template: {config['prompting']['template']}")
    print(f"sampling.mode: {config['sampling']['mode']}")
    print(f"training.epochs: {config['training']['epochs']}")
    print(f"training.bf16: {config['training']['bf16']}")
    print(f"training.fp16: {config['training']['fp16']}")
    print(f"training.eval_batch_size: {config['training']['eval_batch_size']}")
    print(f"scoring.max_completion_batch_size: {config['scoring']['max_completion_batch_size']}")
    print(f"fields: {fields}")


print(f"Repo root: {REPO_ROOT}")
print(f"Drive output dir: {DRIVE_OUTPUT_DIR}")
print(f"Experiment: {EXPERIMENT_ID}")
if DATA_DIR_OVERRIDE:
    print(f"Data override: {DATA_DIR_OVERRIDE}")


Repo root: /content/dl-kaggle-competition-final
Drive output dir: /content/drive/MyDrive/p2p_runs/stage3_stratified_num_choices_task_a100
Experiment: c04_task_choice_stratified


## 5. Smoke Train/Eval

Train a 128-example smoke run for `stratified num choices/task` before spending a full A100 run.


In [10]:
smoke_root = Path(DRIVE_OUTPUT_DIR) / "smoke"
smoke_args = [
    "scripts/run_experiment.py",
    "--experiment", EXPERIMENT_ID,
    "--output-dir", str(smoke_root),
]
extend_with_overrides(smoke_args, with_common_overrides([
    "training.epochs=1",
    "training.bf16=true",
    "training.fp16=false",
    f"training.eval_batch_size={EVAL_BATCH_SIZE}",
    f"scoring.max_completion_batch_size={INFERENCE_COMPLETION_BATCH_SIZE}",
    f"runtime.max_train_examples={SMOKE_TRAIN_EXAMPLES}",
    f"runtime.max_val_examples={SMOKE_VAL_EXAMPLES}",
]))
run_repo_command(smoke_args)
verify_resolved_config(smoke_root)


$ python3 scripts/run_experiment.py --experiment c04_task_choice_stratified --output-dir /content/drive/MyDrive/p2p_runs/stage3_stratified_num_choices_task_a100/smoke --set training.epochs=1 --set training.bf16=true --set training.fp16=false --set training.eval_batch_size=16 --set scoring.max_completion_batch_size=16 --set runtime.max_train_examples=128 --set runtime.max_val_examples=128


KeyboardInterrupt: 

## 6. Full Train/Eval

Train on the train split and evaluate on validation. This is the model-selection run.


In [9]:
full_root = Path(DRIVE_OUTPUT_DIR) / "full"
full_args = [
    "scripts/run_experiment.py",
    "--experiment", EXPERIMENT_ID,
    "--output-dir", str(full_root),
]
extend_with_overrides(full_args, with_common_overrides([
    "training.epochs=1",
    "training.bf16=true",
    "training.fp16=false",
    f"training.eval_batch_size={EVAL_BATCH_SIZE}",
    f"scoring.max_completion_batch_size={INFERENCE_COMPLETION_BATCH_SIZE}",
]))
run_repo_command(full_args)
verify_resolved_config(full_root)


$ python3 scripts/run_experiment.py --experiment c04_task_choice_stratified --output-dir /content/drive/MyDrive/p2p_runs/stage3_stratified_num_choices_task_a100/full --set training.epochs=1 --set training.bf16=true --set training.fp16=false --set training.eval_batch_size=16 --set scoring.max_completion_batch_size=16
LoRA target patterns: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'out_proj']
Matched LoRA modules:
  - model.text_model.layers.0.self_attn.k_proj
  - model.text_model.layers.0.self_attn.o_proj
  - model.text_model.layers.0.self_attn.q_proj
  - model.text_model.layers.0.self_attn.v_proj
  - model.text_model.layers.1.self_attn.k_proj
  - model.text_model.layers.1.self_attn.o_proj
  - model.text_model.layers.1.self_attn.q_proj
  - model.text_model.layers.1.self_attn.v_proj
  - model.text_model.layers.10.self_attn.k_proj
  - model.text_model.layers.10.self_attn.o_proj
  - model.text_model.layers.10.self_attn.q_proj
  - model.text_model.layers.10.self_attn.v_proj
  - model.text_mo

## 7. Final Train+Val Test Prediction

Run only after this ablation wins validation selection. This trains on train+val via `--final-retrain` and predicts the unlabeled test split; it does not train on test labels.


In [ ]:
RUN_FINAL_RETRAIN = False

if RUN_FINAL_RETRAIN:
    final_root = Path(DRIVE_OUTPUT_DIR) / "final_train_val_test"
    final_args = [
        "scripts/run_experiment.py",
        "--experiment", EXPERIMENT_ID,
        "--output-dir", str(final_root),
        "--final-retrain",
        "--predict-test",
    ]
    extend_with_overrides(final_args, with_common_overrides([
        "training.epochs=1",
        "training.bf16=true",
        "training.fp16=false",
        f"training.eval_batch_size={EVAL_BATCH_SIZE}",
        f"scoring.max_completion_batch_size={INFERENCE_COMPLETION_BATCH_SIZE}",
    ]))
    run_repo_command(final_args)
    verify_resolved_config(final_root)
else:
    print("Set RUN_FINAL_RETRAIN = True only after this ablation wins validation selection.")


## 8. Summarize Results

Print the top validation runs from `results/experiments.csv`.


In [ ]:
summary_args = [
    "scripts/summarize_results.py",
    "--sort-by", "val_accuracy",
    "--top", "20",
]
run_repo_command(summary_args)
